Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise: Data Modeling with Apache Iceberg Tables

The fastest query is one that doesn't have to actually read data off disk and all table
formats are designed around this rule. Data modeling in Apache Iceberg is about organizing
our data into files that can be excluded from query processing without actually opening
them up. Every file we can exclude based on partition or metric information (per-file statistics like min/max values, null counts, NaN counts, and record counts that Iceberg stores in its metadata) is saving
time without sacrificing correctness.

In this exercise, you'll learn how to:
- Work with real-world data (NYC Taxi dataset)
- Create tables with different partitioning strategies and Sort Orders
- Analyze table metadata to understand the impact of partitioning on file layout
- Test different query patterns and predicate pushdown, where filter conditions from your SQL queries are evaluated against Iceberg metadata to eliminate files before any data is read
- Use Spark UI to understand query execution


⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Initialize Spark Session

First, let's start our Spark session. This may take a while if starting for the first time as dependencies are downloaded.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("DataModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

Spark 4.0.1 initialized!


/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

## Create Namespace


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")
print("Namespace 'taxi' created!")

Namespace 'taxi' created!


## Download NYC Taxi Data

We'll download NYC Yellow Taxi trip data from **June through October 2023** (5 months). This dataset contains:
- Pick-up and drop-off times and locations
- Trip distances
- Fare amounts
- Payment types
- Passenger counts

**Note**: We're downloading 5 months of data (~225MB total). This will give us enough data to see meaningful differences between partitioning strategies.

In [3]:
import urllib.request
import os
import boto3
from botocore.client import Config

# Configure RustFS client using S3-compatible API
s3_client = boto3.client(
    's3',
    endpoint_url='http://rustfs:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download NYC Yellow Taxi data for June through October 2023 and upload to RustFS
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
rustfs_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

# Create temp directory
os.makedirs(temp_dir, exist_ok=True)

# Download data for months 6-10 (June through October)
months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    rustfs_key = f"{rustfs_prefix}/{filename}"
    
    try:
        # Check if file already exists locally
        if os.path.exists(local_path):
            print(f"{filename} already exists locally, skipping download")
        else:
            # Download file
            print(f"Downloading {filename} (~45MB)...")
            urllib.request.urlretrieve(url, local_path)
            print(f"  Downloaded to {local_path}")
        
        # Upload to RustFS
        print(f"  Uploading to RustFS: s3a://{bucket_name}/{rustfs_key}...")
        s3_client.upload_file(local_path, bucket_name, rustfs_key)
        print(f"  Uploaded successfully!")
        uploaded_files.append(rustfs_key)
        
        # Clean up local file
        os.remove(local_path)
    except Exception as e:
        print(f"  Error processing {filename}: {e}")

print(f"\nAll files ready in RustFS! Total: {len(uploaded_files)} months of data")
print(f"Location: s3a://{bucket_name}/{rustfs_prefix}/")

  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-06.parquet
  Uploading to RustFS: s3a://warehouse/raw/yellow_tripdata_2023-06.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-07.parquet
  Uploading to RustFS: s3a://warehouse/raw/yellow_tripdata_2023-07.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-08.parquet
  Uploading to RustFS: s3a://warehouse/raw/yellow_tripdata_2023-08.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-09.parquet
  Uploading to RustFS: s3a://warehouse/raw/yellow_tripdata_2023-09.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-10.parquet
  Uploading to RustFS: s3a://warehouse/raw/yellow_tripdata_2023-10.parquet...
  Uploaded successfully!

All files ready in RustFS! Total: 5 months of data
Location: s3a://warehouse/raw/


## Load and Explore the Data

Let's load the parquet files and examine their schema and sample data.

In [4]:
# Read all parquet files from RustFS
s3_path = "s3a://warehouse/raw/"

taxi_df = spark.read.parquet(s3_path)

print(f"Reading from: {s3_path}")
print(f"Total records: {taxi_df.count():,}")
print(f"\nSchema:")
taxi_df.printSchema()

Reading from: s3a://warehouse/raw/
Total records: 15,407,558

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
# Show sample data
print("Sample records:")
taxi_df.show(5, truncate=False)

Sample records:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2023-06-01 00:08:48 |2023-06-01 00:29:41  |1              |3.4          |1         |N                 |140         |238         |1           |21.9       |3.5  |0

In [6]:
# Get date range and basic statistics
print("Date range:")
taxi_df.select(
    F.min("tpep_pickup_datetime").alias("min_date"),
    F.max("tpep_pickup_datetime").alias("max_date")
).show()

print("\nBasic statistics:")
taxi_df.select(
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.avg("total_amount"), 2).alias("avg_fare"),
    F.countDistinct(F.to_date("tpep_pickup_datetime")).alias("distinct_days")
).show()

Date range:
+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2002-12-31 22:27:05|2023-11-23 13:40:43|
+-------------------+-------------------+


Basic statistics:
+-----------+------------+--------+-------------+
|total_trips|avg_distance|avg_fare|distinct_days|
+-----------+------------+--------+-------------+
|   15407558|        4.35|   29.05|          161|
+-----------+------------+--------+-------------+



## Creating Iceberg Tables with Different Partitioning Specifications

Iceberg table partitioning is done using a series of partition transform functions which we call a "spec" (short for "partition specification"), the set of transforms that tells Iceberg how to assign rows to files. In older table formats, you add synthetic partition columns to your data and manage their values yourself (e.g., `PARTITIONED BY (year STRING, month STRING)`). Iceberg instead applies transform functions (like `months()`) to existing columns. No extra columns needed, and the partition values are tracked in metadata, not directory names. Because the transforms are well-defined, query engines can automatically match your filter predicates to the partition scheme even when your query doesn't reference the transform directly. This is called **hidden partitioning**: the partition scheme is invisible to queries, so you filter on the original column and Iceberg handles the transform internally.

An Iceberg table can have multiple active specs at the same time but for this exercise we'll use a single spec per table to show the different effects partitioning has on data file size and query performance. We'll use CREATE TABLE as SELECT (CTAS) Statements to copy the existing Taxi data into new files using our specified partitioning. This is an expensive approach to creating Iceberg tables from parquet files and we'll go over other methods in future modules.

*Note: We are setting the default target file size to 16MB for these tables in order to better illustrate the effects of partitioning and pruning*

## Strategy 1: Unpartitioned Table

First, let's create a baseline table without partitioning

In [7]:
# Create unpartitioned table using CTAS
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_unpartitioned
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Unpartitioned table created!")

Unpartitioned table created!


### Analyze Unpartitioned Table Metadata

Iceberg provides [**metadata tables**](https://iceberg.apache.org/docs/latest/spark-queries/#inspecting-tables) that let you inspect the physical layout without scanning data. 
In every engine these are accessed differently. Spark uses a "catalog.namespace.table.metadata_table" pattern.

In [8]:
# Check the files metadata table
# readable_metrics is an Iceberg metadata column containing per-file statistics
# (lower_bound, upper_bound, null_count, nan_count, etc.) for every data column.
# These stats let query engines skip files without opening them.
print("Files in unpartitioned table:")
spark.sql("""
    SELECT 
        file_path,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count,
        readable_metrics.total_amount.lower_bound as total_amount_min,
        readable_metrics.total_amount.upper_bound as total_amount_max
    FROM polaris.taxi.trips_unpartitioned.files
""").show(truncate=False)

Files in unpartitioned table:
+----------------------------------------------------------------------------------------------------------+-------------------+------------+----------------+----------------+
|file_path                                                                                                 |size_mb            |record_count|total_amount_min|total_amount_max|
+----------------------------------------------------------------------------------------------------------+-------------------+------------+----------------+----------------+
|s3://warehouse/taxi/trips_unpartitioned/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00001.parquet|16.06031608581543  |1059000     |-561.0          |981.6           |
|s3://warehouse/taxi/trips_unpartitioned/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a8-0-00002.parquet|15.952461242675781 |1059000     |-556.55         |386987.63       |
|s3://warehouse/taxi/trips_unpartitioned/data/00000-25-83001fac-d39a-4671-ac58-ba8d3d3e00a

In [9]:
# The entries metadata table exposes manifest-level entries (one row per data file
# per manifest). It contains the same information as the files table but parallelizes
# better in Spark because each manifest can be read independently.
spark.sql("""
    SELECT 
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_unpartitioned.entries
""").show()

+------------+----------+
|record_count|data_files|
+------------+----------+
|    15407558|        18|
+------------+----------+



In [10]:
# Check snapshots
print("Snapshots:")
spark.sql("""
    SELECT 
        committed_at,
        snapshot_id,
        operation
    FROM polaris.taxi.trips_unpartitioned.snapshots
""").show(truncate=False)

Snapshots:
+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-17 10:37:16.647|2506011983532340810|append   |
+-----------------------+-------------------+---------+



### Check RustFS for Unpartitioned Table

**RustFS Location**: http://localhost:9001/browser/warehouse/taxi/trips_unpartitioned/

*Note: Use username `admin` and password `password` to login to RustFS.*

The data directory should contain all the files referenced in the metadata table above. 
The metadata directory has the metadata.json, manifests (*m*.avro) and manifest_list (snap*.avro) files. 

## Strategy 2: Monthly Partitioning

Now let's partition by **month**. Iceberg supports temporal transforms like `month()` that automatically extract the month from a timestamp and store
the value in the table metadata. When partitioning is set in Iceberg, all new files written to the table will all have the same value for the output of the partition transform. That means in this case, data files we create will only have a single month in each of them. This means queries based on time only have to read data files with the same "month" as required by predicates of the query.

In [11]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_month
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Monthly partitioned table created!")

Monthly partitioned table created!


### Analyze Monthly Partitioned Table

In [12]:
print("Files in monthly partitioned table:")
spark.sql("""
    SELECT 
        file_path,
        partition,
        file_size_in_bytes / 1024 / 1024 as size_mb,
        record_count
    FROM polaris.taxi.trips_by_month.files
""").show(truncate=False)

Files in monthly partitioned table:
+-----------------------------------------------------------------------------------------------------------------------------------------+---------+---------------------+------------+
|file_path                                                                                                                                |partition|size_mb              |record_count|
+-----------------------------------------------------------------------------------------------------------------------------------------+---------+---------------------+------------+
|s3://warehouse/taxi/trips_by_month/data/tpep_pickup_datetime_month=2023-11/00000-124-4f0fa3d8-0c2e-4595-9444-3b5c6ea905b9-0-00004.parquet|{646}    |0.006367683410644531 |13          |
|s3://warehouse/taxi/trips_by_month/data/tpep_pickup_datetime_month=2023-07/00000-124-4f0fa3d8-0c2e-4595-9444-3b5c6ea905b9-0-00001.parquet|{642}    |16.1483097076416     |1038000     |
|s3://warehouse/taxi/trips_by_month/dat

In [13]:
# Check partition statistics
# Note we are using the "entries" table because it parallelizes better on Spark than the "partitions" table
# The partition values below (e.g., 641, 642) are month offsets from epoch (Jan 1970 = 0).
# So 641 = June 2023, 642 = July 2023, etc. The next cell explains this encoding.
print("Partition statistics:")
spark.sql("""
    SELECT 
        data_file.partition,
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_month.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
""").show()

Partition statistics:
+---------+------------+----------+
|partition|record_count|data_files|
+---------+------------+----------+
|    {395}|           5|         1|
|    {396}|           1|         1|
|    {467}|          10|         1|
|    {468}|          11|         1|
|    {640}|          19|         1|
|    {641}|     3307234|         4|
|    {642}|     2907084|         3|
|    {643}|     2824199|         3|
|    {644}|     2846740|         3|
|    {645}|     3522242|         4|
|    {646}|          13|         1|
+---------+------------+----------+



### Check RustFS for Monthly Partitioned Table

**RustFS Location**: http://localhost:9001/browser/warehouse/taxi/trips_by_month/

**Understanding partition values**: The `months()` transform outputs an integer representing the number of months since epoch (January 1970 = month 0). So `{641}` = month 641 = June 2023, `{642}` = July 2023, and so on. This is an internal representation; query engines translate it automatically, so you never need to use these numbers in your queries.

**Important Note**: The partition directories you see (e.g., `tpep_pickup_datetime_month=2023-06/`) are a convenience for humans browsing the storage. Iceberg's actual partition tracking is metadata-driven: it stores partition values in manifest files, not in directory paths. The directories exist because Iceberg writes data files with these prefixes for human readers but the structure is not used during query planning.

## Strategy 3: Daily Partitioning

We can also use the "days" transform to partition our data into files where every row shares a single value for day. This will end up generating much smaller files (1/30th the size of our monthly tables) but they will be much more selective. Most engines currently do not perform well with small files, so this is generally too small for good performance, but we will address this in later modules.

In [14]:
# Create daily partitioned table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_by_day
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Daily partitioned table created!")

Daily partitioned table created!


### Analyze Daily Partitioned Table

In [15]:
# Check partition statistics - note we have many more partitions now
print("Partition statistics (showing first 10):")
spark.sql("""
    SELECT 
        data_file.partition,
        SUM(data_file.record_count) as record_count,
        COUNT(*) as data_files
    FROM polaris.taxi.trips_by_day.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
    LIMIT 10
""").show()

Partition statistics (showing first 10):
+------------+------------+----------+
|   partition|record_count|data_files|
+------------+------------+----------+
|{2002-12-31}|           5|         1|
|{2003-01-01}|           1|         1|
|{2008-12-31}|          10|         1|
|{2009-01-01}|          11|         1|
|{2023-05-31}|          19|         1|
|{2023-06-01}|      123258|         1|
|{2023-06-02}|      118971|         1|
|{2023-06-03}|      118159|         1|
|{2023-06-04}|      100293|         1|
|{2023-06-05}|      103470|         1|
+------------+------------+----------+



In [16]:
# Count total partitions
partition_count = spark.sql("""
    SELECT COUNT(DISTINCT data_file.partition) as partition_count 
    FROM polaris.taxi.trips_by_day.entries
""").collect()[0][0]

print(f"Total partitions in daily partitioned table: {partition_count}")

Total partitions in daily partitioned table: 161


In [17]:
# Check file distribution across partitions
print("File count per partition (sample):")
spark.sql("""
    SELECT 
        data_file.partition,
        COUNT(*) as file_count,
        SUM(data_file.record_count) as total_records,
        ROUND(SUM(data_file.file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.taxi.trips_by_day.entries
    GROUP BY data_file.partition
    ORDER BY data_file.partition
    LIMIT 10
""").show()

File count per partition (sample):
+------------+----------+-------------+-------------+
|   partition|file_count|total_records|total_size_mb|
+------------+----------+-------------+-------------+
|{2002-12-31}|         1|            5|         0.01|
|{2003-01-01}|         1|            1|         0.01|
|{2008-12-31}|         1|           10|         0.01|
|{2009-01-01}|         1|           11|         0.01|
|{2023-05-31}|         1|           19|         0.01|
|{2023-06-01}|         1|       123258|         1.85|
|{2023-06-02}|         1|       118971|          1.8|
|{2023-06-03}|         1|       118159|         1.79|
|{2023-06-04}|         1|       100293|         1.59|
|{2023-06-05}|         1|       103470|         1.61|
+------------+----------+-------------+-------------+



### Check RustFS for Daily Partitioned Table

**RustFS Location**: http://localhost:9001/browser/warehouse/taxi/trips_by_day/

## What shouldn't we be partitioning on?

In general, we want to make sure our files are all close to our target file size so picking a very high cardinality column (a column with many unique values) is usually not
good for partitioning. While picking out a partitioning spec, remember that query cost scales with the number of files and the amount of data being read. In many systems, 
having many small files will perform worse than a single larger file. As a rough guideline, aim for **100 MB – 1 GB per partition**; files under a few MB start to hurt because the per-file overhead (opening, reading metadata, closing) dominates the actual data read time. Notice above that daily partitioning produced ~1.8 MB files, which is well under the ideal range.

## Strategy 4: Monthly Partition with Sort Order

Iceberg also supports the concept of a "Sort Order" which tells query engines how data should be laid out before it is written to the table. Sort orders control the physical layout of data within files to speed up queries that filter on the sorted columns. Since Iceberg can exclude a file if *any* column metric can prove that the query will not need to read the file, having query columns
sorted is often a good speedup.

In this example let's create a table where we are partitioned on month and then sorted on the pickup location.

In [2]:
# Create monthly partitioned table with sort order on pickup location
# Important: Set sort order BEFORE inserting data so it applies during write
#
# Spark SQL doesn't support setting sort order in a CTAS statement, so we use a
# three-step workaround: create empty table, set sort order, then insert data.
# This is a Spark limitation, not a fundamental Iceberg pattern.

# Step 1: Create empty table structure (using CTAS with false condition)
spark.sql(""" DROP TABLE IF EXISTS polaris.taxi.trips_sorted """)
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.taxi.trips_sorted
    USING iceberg
    PARTITIONED BY (months(tpep_pickup_datetime))
    TBLPROPERTIES (
        'format-version' = '3',
        'write.target-file-size-bytes' = '16777216'
    )
    AS SELECT * FROM parquet.`s3a://warehouse/raw/`
    WHERE 1 = 0
""")

# Step 2: Set the sort order (must be done before data is written)
spark.sql("""
    ALTER TABLE polaris.taxi.trips_sorted
    WRITE ORDERED BY PULocationID
""")

print("Table structure created with sort order configured")

# Step 3: Insert data - it will be sorted as it's written
spark.sql("""
    INSERT INTO polaris.taxi.trips_sorted
    SELECT * FROM parquet.`s3a://warehouse/raw/`
""")

print("Sorted table populated!")

Table structure created with sort order configured
Sorted table populated!


### Analyze Sorted Table

The key benefit of sorting is visible in the min/max bounds of each file, each file covers an exclusive range of pickup locations. This is great for location based queries like trying to find all the rides which occurred at a specific location. There is a cost to sorting which increases the amount of compute and time required to insert data into the table. Iceberg also does not re-sort all data on subsequent inserts so table maintenance procedures are required to maintain sort efficiency.

In [3]:
# Check file metadata with bounds
print("File statistics (showing readable metrics including min/max for PULocationID):")
spark.sql("""
    SELECT 
        file_path,
        partition,
        record_count,
        readable_metrics.PULocationID.lower_bound as PULocationID_MIN,
        readable_metrics.PULocationID.upper_bound as PULocationID_MAX
    FROM polaris.taxi.trips_sorted.files
    ORDER BY partition DESC, PULocationID_MIN
    LIMIT 10
""").show(truncate=False)

File statistics (showing readable metrics including min/max for PULocationID):
+---------------------------------------------------------------------------------------------------------------------------------------+---------+------------+----------------+----------------+
|file_path                                                                                                                              |partition|record_count|PULocationID_MIN|PULocationID_MAX|
+---------------------------------------------------------------------------------------------------------------------------------------+---------+------------+----------------+----------------+
|s3://warehouse/taxi/trips_sorted/data/tpep_pickup_datetime_month=2023-11/00010-172-1e5d68a9-834f-4ca3-a5bd-299b9869611a-0-00002.parquet|{646}    |13          |43              |265             |
|s3://warehouse/taxi/trips_sorted/data/tpep_pickup_datetime_month=2023-10/00008-170-1e5d68a9-834f-4ca3-a5bd-299b9869611a-0-00002.parquet|{645

## Query Testing: Understanding Pushdown Optimizations

Now let's test different query patterns to see how partitioning and metadata affect performance. Iceberg takes predicates that are given to it from Spark and evaluates them against
metadata to determine which files will actually be scanned.

Recall from E1.1 that Iceberg's metadata forms a tree: **snapshot → manifest list → manifests → data files**. A manifest is a metadata file that lists a group of data files along with their partition values and column-level statistics (min, max, null count). Why the extra layer? Grouping files into manifests lets Iceberg skip entire groups at once using partition summaries, which is critical at scale with thousands of files. During query planning, Iceberg reads manifests to decide which data files to scan. This is where file skipping happens.

* The partition related predicates are compared to the manifest partition summaries and if possible we can then skip every file in that manifest.
* Manifests that pass the first filter are opened and each of the datafile entries within is compared against both partition related predicates and column related predicates.


### Query 1: No Predicate Pushdown (Full Scan)

This query has no filters, so all data must be read.

In [2]:
# Query without any filters - full scan
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
""")

print("Full scan query results:")
result.show()


Full scan query results:
+-----------+--------+--------------+
|total_trips|avg_fare| total_revenue|
+-----------+--------+--------------+
|   15407558|   29.05|4.4756965321E8|
+-----------+--------+--------------+



**Check Spark UI: http://localhost:4040/SQL/**

1. Go to the **SQL** tab - find this query
2. Click on the **Description** link
3. Click on the **showString** link (the last operation)
4. Look for **BatchScan polaris.taxi.trips_by_day** section
4. Examine the Iceberg-specific statistics in the BatchScan relation:
   - `number of result data files`: Total data files read (should be ALL files in the table)
   - `number of skipped data files`: Should be 0 (no files were skipped)
   - `number of skipped data manifests`: Should be 0 (all manifests were scanned)
   - `total data file size (bytes)`: Total bytes scanned
   - `number of output rows`: Total rows returned from the scan

Iceberg can't do any pushdowns in this case so every single file is read.

### Query 2: Partition Pushdown

This query filters by date, which allows Iceberg to skip entire partitions.

In [ ]:
# Query with date filter - partition pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as total_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(SUM(total_amount), 2) as total_revenue
    FROM polaris.taxi.trips_by_day
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
""")

print("Partition pushdown query results (August 2023 only):")
result.show()


**Check Spark UI: http://localhost:4040/SQL/**

1. Go to the **SQL** tab - find this query
2. Click on the **Description** link
3. Click on the **showString** link (the last operation)
4. Look for **BatchScan polaris.taxi.trips_by_day** section
5. Check the Iceberg statistics:
   - `number of result data files`: Only files for August (~31 files, one per day)
   - `number of skipped data files`: Files for other months that were skipped
   - `number of skipped data manifests`: Manifests for other partitions skipped
   - `total data file size (bytes)`: Much smaller than full scan
   - `total planning duration (ms)`: Time Iceberg took to plan the query

**Partition pruning skipped ~4/5 of the files based on partition metadata!**

### Query 3: Metric Pushdown (Min/Max Filtering)

This query filters by a non-partition column with a range predicate. Iceberg can use file-level min/max statistics to skip files.

In [ ]:
# Query with location filter - metric pushdown
result = spark.sql("""
    SELECT 
        COUNT(*) as trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE PULocationID IN (132, 138, 161, 230, 237)
""")

print("Metric pushdown query results (specific locations):")
result.show()


**Check Spark UI: http://localhost:4040/SQL/**

1. Go to the **SQL** tab - find this query
2. Click on the **Description** link
3. Click on the **showString** link
4. Look for **BatchScan polaris.taxi.trips_sorted** section
5. Check the statistics:
   - `number of result data files`: Fewer files than total in table
   - `number of skipped data files`: Files skipped because PULocationID min/max stats don't overlap with filter
   - `number of scanned data manifests`: Manifests that were examined for file statistics
   - `total data file size (bytes)`: Reduced due to metric-based filtering

**Sort order + metric pushdown: Files skipped using min/max bounds on PULocationID!**

### Query 4: Combined Partition + Metric Pushdown

This query combines both: partition pruning (by month) and metric filtering (by location).

In [ ]:
# Query with both month and location filters
result = spark.sql("""
    SELECT 
        COUNT(*) as matching_trips,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_distance), 2) as avg_distance
    FROM polaris.taxi.trips_sorted
    WHERE tpep_pickup_datetime >= '2023-08-01'
      AND tpep_pickup_datetime < '2023-09-01'
      AND PULocationID IN (132, 138, 161, 230, 237)
""")

print("Combined pushdown query results (August 2023, specific locations):")
result.show()


**Check Spark UI: http://localhost:4040/SQL/**

1. Go to the **SQL** tab - find this query
2. Click on the **Description** link
3. Click on the **showString** link
4. Look for **BatchScan polaris.taxi.trips_sorted** section
5. Check the statistics:
   - `number of result data files`: Minimal - only files that match BOTH filters
   - `number of skipped data files`: Maximum skipping (partition + metrics)
   - `number of skipped data manifests`: Manifests eliminated by partition filter
   - `total data file size (bytes)`: Smallest scan - both optimizations applied
   - `total planning duration (ms)`: Iceberg's metadata-based planning time

**Best case: Partition pruning eliminated months, then metric pushdown eliminated files within August!**

## Performance Comparison: All Partitioning Strategies

Let's run the same query against all our table versions to compare performance. Here we are setting up a function that will just run a query against all of the tables
we made above and show the performance. Actually understanding the performance is a little more complicated because we have a mix of a cost from having to open files
and a cost for reading those files. This means in some cases two queries may read the same amount of data but the one with small files will perform worse. This sort of
performance characteristic is very engine dependent.

**You may want to run queries several times to try to avoid any warmup costs**



In [3]:
import time

def compare_partitioning_strategies(where_clause, description):
    """
    Compare query performance across all partitioning strategies.
    
    Args:
        where_clause: SQL WHERE clause (without the WHERE keyword)
        description: Human-readable description of the query
    """
    # List of all tables to compare
    tables = [
        ('trips_unpartitioned', 'Unpartitioned'),
        ('trips_by_month', 'Monthly Partitions'),
        ('trips_by_day', 'Daily Partitions'),
        ('trips_sorted', 'Monthly + Sorted by Location')
    ]
    
    results = []
    
    print(f"Query: {description}")
    print(f"Predicate: {where_clause}")
    print("\n" + "=" * 80)
    
    for table_name, table_desc in tables:
        # Build the query with the table name substituted
        query = f"""
            SELECT COUNT(*) as trip_count,
                   ROUND(AVG(total_amount), 2) as avg_fare
            FROM polaris.taxi.{table_name}
            WHERE {where_clause}
        """
        
        start = time.time()
        result = spark.sql(query).collect()
        elapsed = time.time() - start
        
        count = result[0][0]
        avg_fare = result[0][1] if result[0][1] else 0.0
        results.append((table_desc, count, avg_fare, elapsed))
        
        print(f"{table_desc:30} | Trips: {count:>6,} | Avg: ${avg_fare:>6.2f} | Time: {elapsed:>6.3f}s")
    
    print("=" * 80)
    
    # Calculate and display speedups
    baseline_time = results[0][3]
    print("\nSpeedups vs Unpartitioned:")
    for table_desc, count, avg_fare, elapsed in results[1:]:
        speedup = baseline_time / elapsed if elapsed > 0 else 0
        print(f"  {table_desc:30} : {speedup:>5.2f}x faster")
    
    print("\n" + "=" * 80 + "\n")
    
    return results

Use `compare_partitioning_strategies(where_clause, description)` to run the same query against all four table versions and compare their performance.

### Example 1: Time Range + Location

Query for 3 days in mid-August with a specific pickup location. Even though our tables are using different partitioning
specs, Iceberg is able to convert this predicate to match each table.

In [4]:
# Example 1: 3 days + specific location
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-08-14' AND tpep_pickup_datetime < '2023-08-17' AND PULocationID = 237",
    description="3 days in mid-August, pickup location 237"
)

Query: 3 days in mid-August, pickup location 237
Predicate: tpep_pickup_datetime >= '2023-08-14' AND tpep_pickup_datetime < '2023-08-17' AND PULocationID = 237

Unpartitioned                  | Trips: 13,561 | Avg: $ 19.49 | Time:  1.056s
Monthly Partitions             | Trips: 13,561 | Avg: $ 19.49 | Time:  0.376s
Daily Partitions               | Trips: 13,561 | Avg: $ 19.49 | Time:  0.188s
Monthly + Sorted by Location   | Trips: 13,561 | Avg: $ 19.49 | Time:  0.223s

Speedups vs Unpartitioned:
  Monthly Partitions             :  2.81x faster
  Daily Partitions               :  5.63x faster
  Monthly + Sorted by Location   :  4.73x faster




[('Unpartitioned', 13561, 19.49, 1.0559840202331543),
 ('Monthly Partitions', 13561, 19.49, 0.3761940002441406),
 ('Daily Partitions', 13561, 19.49, 0.18764925003051758),
 ('Monthly + Sorted by Location', 13561, 19.49, 0.22310113906860352)]

**Analysis:**
- **Unpartitioned**: Scans all ~153 files
- **Monthly**: Scans all August files
- **Daily**: Scans only 3 files 
- **Monthly + Sorted**: Scans a single August file

### Example 2: Full Month Range

Query an entire month to see partition pruning without day-level granularity.

In [5]:
# Example 2: Full month
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-07-01' AND tpep_pickup_datetime < '2023-08-01'",
    description="All of July 2023"
)

Query: All of July 2023
Predicate: tpep_pickup_datetime >= '2023-07-01' AND tpep_pickup_datetime < '2023-08-01'

Unpartitioned                  | Trips: 2,907,084 | Avg: $ 28.57 | Time:  1.147s
Monthly Partitions             | Trips: 2,907,084 | Avg: $ 28.57 | Time:  0.293s
Daily Partitions               | Trips: 2,907,084 | Avg: $ 28.57 | Time:  0.298s
Monthly + Sorted by Location   | Trips: 2,907,084 | Avg: $ 28.57 | Time:  0.175s

Speedups vs Unpartitioned:
  Monthly Partitions             :  3.92x faster
  Daily Partitions               :  3.85x faster
  Monthly + Sorted by Location   :  6.55x faster




[('Unpartitioned', 2907084, 28.57, 1.1473119258880615),
 ('Monthly Partitions', 2907084, 28.57, 0.2929720878601074),
 ('Daily Partitions', 2907084, 28.57, 0.29838013648986816),
 ('Monthly + Sorted by Location', 2907084, 28.57, 0.17508292198181152)]

**Analysis:**
- Monthly and Monthly + Sorted both benefit equally from monthly partition pruning. They scan the same number of files for a full-month query. The sort order on `PULocationID` adds **no benefit here** because this query has no predicate on the sort column; any timing difference between the two is noise or minor file layout variation.
- Daily partitioning prunes less effectively for a full-month query because it still has to open ~31 small files (one per day), which adds per-file overhead.
- Both partitioned versions do much better than unpartitioned.

### Example 3: Single Day + Location Range

Very selective query: one day, multiple locations. Shows metric pushdown effectiveness.

In [6]:
# Example 3: Single day + location range
compare_partitioning_strategies(
    where_clause="tpep_pickup_datetime >= '2023-08-15' AND tpep_pickup_datetime < '2023-08-16' AND PULocationID IN (132, 138, 161)",
    description="Single day (Aug 15), 3 specific locations"
)

Query: Single day (Aug 15), 3 specific locations
Predicate: tpep_pickup_datetime >= '2023-08-15' AND tpep_pickup_datetime < '2023-08-16' AND PULocationID IN (132, 138, 161)

Unpartitioned                  | Trips: 14,077 | Avg: $ 52.70 | Time:  0.581s
Monthly Partitions             | Trips: 14,077 | Avg: $ 52.70 | Time:  0.239s
Daily Partitions               | Trips: 14,077 | Avg: $ 52.70 | Time:  0.134s
Monthly + Sorted by Location   | Trips: 14,077 | Avg: $ 52.70 | Time:  0.229s

Speedups vs Unpartitioned:
  Monthly Partitions             :  2.43x faster
  Daily Partitions               :  4.34x faster
  Monthly + Sorted by Location   :  2.53x faster




[('Unpartitioned', 14077, 52.7, 0.5811936855316162),
 ('Monthly Partitions', 14077, 52.7, 0.23889708518981934),
 ('Daily Partitions', 14077, 52.7, 0.13391947746276855),
 ('Monthly + Sorted by Location', 14077, 52.7, 0.2293853759765625)]

**Analysis:**
- Unpartitioned still benefits a bit here from column metric restrictions even though it is not explicitly partitioned.
- Monthly still requires all monthly files to be read
- Daily partitions perform best due to most selective partition pruning.

### Example 4: Location-Only Filter (No Time)

Query without a time predicate. Shows the importance of partitioning on the query column.

In [7]:
# Example 4: Location only (no time filter)
compare_partitioning_strategies(
    where_clause="PULocationID = 237",
    description="All trips from location 237 (no time filter)"
)

Query: All trips from location 237 (no time filter)
Predicate: PULocationID = 237

Unpartitioned                  | Trips: 694,148 | Avg: $ 20.53 | Time:  0.441s
Monthly Partitions             | Trips: 694,148 | Avg: $ 20.53 | Time:  0.388s
Daily Partitions               | Trips: 694,148 | Avg: $ 20.53 | Time:  0.810s
Monthly + Sorted by Location   | Trips: 694,148 | Avg: $ 20.53 | Time:  0.160s

Speedups vs Unpartitioned:
  Monthly Partitions             :  1.14x faster
  Daily Partitions               :  0.55x faster
  Monthly + Sorted by Location   :  2.76x faster




[('Unpartitioned', 694148, 20.53, 0.44139981269836426),
 ('Monthly Partitions', 694148, 20.53, 0.3878641128540039),
 ('Daily Partitions', 694148, 20.53, 0.8096134662628174),
 ('Monthly + Sorted by Location', 694148, 20.53, 0.1602175235748291)]

**Analysis:**
- Only column metric pruning is available
- **Sorted table** has major benefits from metric pushdown on PULocationID
- Daily table ends up slower than unpartitioned because of file opening cost

### Try Your Own Query!

Use the function to test different query patterns. Experiment with:
- Different time ranges (hours, days, weeks, months)
- Different location filters
- Other columns like `total_amount`, `trip_distance`, `passenger_count`

In [ ]:
# Your custom query here!
# Uncomment and modify:

# compare_partitioning_strategies(
#     where_clause="tpep_pickup_datetime >= '2023-09-01' AND tpep_pickup_datetime < '2023-09-02' AND total_amount > 50",
#     description="High-value trips on Sept 1"
# )

### Key Takeaways from Performance Comparisons

1. **Hidden Partitioning**: As introduced above, we never had to specify our partition transform in our queries. Iceberg translated our raw column predicates to match the partition scheme automatically
2. **Number of Partitions**: Helpful for specific queries but small files can cause performance issues for non-selective queries
3. **Sort Orders Enable Metric Pushdown**: Sorted columns allow file-level filtering even without partitioning on that column
4. **Partitioning and Sort Order are Query Dependent**: If our query isn't using a partition transform or column metric, it won't benefit from pushdowns

**Check the Spark UI** (http://localhost:4040/SQL/) for each query to see the actual file statistics!

## Summary and Best Practices

### What We Learned:

1. **Partitioning Strategies**:
   - Consider column cardinality
   - Match common query patterns
   - Transforms allow flexibility in partitioning

2. **Sort Orders**:
   - Help with range queries and metric pushdown
   - Clusters related data together
   - Use for columns frequently used in WHERE clauses

3. **Metadata Tables**:
   - `files`: See physical file layout and statistics
   - `snapshots`: Track table commits

4. **Query Optimization**:
   - Partition pruning: Skip entire manifests or files
   - Metric pushdown: Skip files using min/max statistics
  
### Best Practices:

- **Partition by query patterns**: Choose partitions based on common filters
- **Avoid over-partitioning**: Too many small files hurt performance
- **Monitor partition sizes**: Aim for 100MB - 1GB per partition
- **Use Iceberg's hidden partitioning**: No need to include partition columns in queries
- **Leverage sort orders**: For range queries on non-partition columns
- **Check metadata tables**: Before and after changes to understand impact

### Further Exploration:

- Try different partition strategies for your use cases
- Experiment with multi-column partitioning
- Compare performance across different engines (Spark vs Trino)